# KNN Probe Baseline — Serafim PT-PT (900M sentence-encoder)

Frozen-encoder KNN-probe baseline for `PORTULAN/serafim-900m-portuguese-pt-sentence-encoder` (`src.config.MODEL_REGISTRY["serafim_900m"]`).

## Section 1 — Setup & Data Loading

Imports, the pinned `serafim_900m` entry from `src.config.MODEL_REGISTRY`.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import torch

sys.path.insert(0, str(Path("../..").resolve()))
from src.capture import (
    capture_full_attention_subset,
    capture_hidden_states_and_mean_pooled,
    pool_mean,
)
from src.config import MODEL_REGISTRY
from src.data import build_datasets, load_label_map, load_splits
from src.evaluate import evaluate_predictions, full_test_report
from src.knn_probe import (
    extract_mean_pooled_embeddings,
    history_to_dicts,
    predict_full_proba,
    train_knn_probe,
    training_args_payload,
)
from src.linear_probe import build_frozen_encoder_and_tokenizer
from src.plotting import plot_confusion_matrix
from src.runtime import get_device, set_all_seeds

set_all_seeds()

In [ ]:
model_cfg = MODEL_REGISTRY["serafim_900m"]

df = load_splits()
LABELS = load_label_map()["labels"]

device = get_device()
print(f"device: {device}")
print(f"split sizes: {df['split'].value_counts().to_dict()}")
print(f"labels ({len(LABELS)}): {LABELS}")

## Section 2 — Step 1: Load the Frozen Encoder

`build_frozen_encoder_and_tokenizer`.

In [ ]:
model, tokenizer = build_frozen_encoder_and_tokenizer(model_cfg, device)
model

## Section 3 — Step 2: The Forward Pass and Tuple Extraction

Wrapped in `torch.no_grad()` since the encoder is frozen and this is inference, not training.

In [ ]:
sample_texts = df["Sumario"].iloc[:4].tolist()

encoded = tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    max_length=model_cfg.max_length,
    return_tensors="pt",
).to(device)

with torch.no_grad():
    outputs = model(**encoded)

hidden_states = outputs.hidden_states
print(f"num hidden_states tensors: {len(hidden_states)}")
print(f"shape of each tensor: {hidden_states[0].shape}")

## Section 4 — Step 3: Mean-Pool the Last Hidden Layer

Serafim's pretraining objective is a sentence embedding, not a `[CLS]` classification token.

In [ ]:
last_hidden_state = hidden_states[-1]
attention_mask = encoded["attention_mask"]

pooled = pool_mean(last_hidden_state, attention_mask)
pooled.shape

## Section 5 — Step 4: Package Steps 2–3 Into a Reusable Function

`embed_sumarios` packages Steps 2–3 (tokenize → forward pass → mean pooling) into one reusable function for this notebook's demo cells.

In [ ]:
def embed_sumarios(texts: list[str]) -> torch.Tensor:
    """Steps 2-3 end to end: tokenize -> forward pass -> mean pooling over real tokens.

    Returns a (len(texts), hidden_size) tensor.
    """
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=model_cfg.max_length,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**encoded)

    return pool_mean(outputs.hidden_states[-1], encoded["attention_mask"])

embed_sumarios(sample_texts).shape

## Section 6 — Step 5: Extract Cached Embeddings & Model-Select the KNN Probe

`src.data.build_datasets` tokenizes `Sumario` and attaches the numeric `label_id` for every row.

In [ ]:
datasets = build_datasets(df, tokenizer, max_length=model_cfg.max_length)
{split: len(ds) for split, ds in datasets.items()}

In [ ]:
extract_batch_size = model_cfg.batch_size(device.type)

train_embeddings = extract_mean_pooled_embeddings(
    model, datasets["train"], device, batch_size=extract_batch_size
).numpy()
val_embeddings = extract_mean_pooled_embeddings(
    model, datasets["val"], device, batch_size=extract_batch_size
).numpy()

print(f"train_embeddings: {train_embeddings.shape}")
print(f"val_embeddings  : {val_embeddings.shape}")

In [ ]:
KNN_NEIGHBORS_GRID = (3, 5, 7, 9, 11, 15, 21, 31)
KNN_WEIGHTS = "distance"
KNN_METRIC = "cosine"

knn, knn_history, best_log = train_knn_probe(
    train_embeddings, datasets["train"].label_ids,
    val_embeddings, datasets["val"].label_ids,
    labels=LABELS,
    n_neighbors_grid=KNN_NEIGHBORS_GRID, weights=KNN_WEIGHTS, metric=KNN_METRIC,
)
best_log

## Section 7 — Full Test-Split Results, XAI Artifacts & Metrics Report

`src.capture.capture_hidden_states_and_mean_pooled` runs the frozen encoder (no classifier attached) over the full test split to save.

In [ ]:
test_df = df[df["split"] == "test"].reset_index(drop=True)

results_dir = model_cfg.results_dir("knn_probe")
batch_size = model_cfg.batch_size(device.type)

full_split_artifacts = capture_hidden_states_and_mean_pooled(model, datasets["test"], device, batch_size=batch_size)
for name, array in full_split_artifacts.items():
    np.save(results_dir / f"{name}.npy", array)
    print(f"{name}.npy: {array.shape}")

In [ ]:
test_embeddings = full_split_artifacts["cls_embeddings"]

probs = predict_full_proba(knn, test_embeddings, num_labels=len(LABELS))
logits = np.log(probs + 1e-12)

np.save(results_dir / "probs.npy", probs)
np.save(results_dir / "logits.npy", logits)
print(f"probs.npy: {probs.shape}")
print(f"logits.npy: {logits.shape}")

In [ ]:
N_ATTENTION_SUBSET = 50

attention_artifacts = capture_full_attention_subset(
    model, datasets["test"], device, n_samples=N_ATTENTION_SUBSET
)
for name, array in attention_artifacts.items():
    np.save(results_dir / f"{name}.npy", array)
    print(f"{name}.npy: {array.shape}")

attention_subset_ids = test_df["ImportJurisprudenciaId"].iloc[:N_ATTENTION_SUBSET].tolist()
with open(results_dir / "attention_subset_ids.json", "w", encoding="utf-8") as f:
    json.dump(attention_subset_ids, f, indent=2)

In [ ]:
test_label_ids = test_df["label_id"].tolist()
test_pred_ids = logits.argmax(axis=-1).tolist()

test_metrics = evaluate_predictions(test_label_ids, test_pred_ids, LABELS)
test_report = full_test_report(test_label_ids, test_pred_ids, LABELS)

test_metrics

In [ ]:
checkpoint_dir = results_dir / "checkpoint"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(knn, checkpoint_dir / "knn_classifier.joblib")
tokenizer.save_pretrained(checkpoint_dir)

results_df = test_df[["ImportJurisprudenciaId", "Decisao"]].copy()
results_df["predicted_label"] = [LABELS[i] for i in test_pred_ids]
results_df["confidence"] = probs.max(axis=-1).tolist()
results_df.to_csv(results_dir / "results.csv", index=False)

metrics_payload = {
    "neighbor_grid_history": history_to_dicts(knn_history),
    "summary": test_metrics,
    "classification_report": test_report["classification_report"],
    "confusion_matrix": test_report["confusion_matrix"],
    "confusion_matrix_labels": test_report["confusion_matrix_labels"],
}
with open(results_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

plot_confusion_matrix(
    test_report["confusion_matrix"], LABELS, results_dir / "confusion_matrix.png",
    title=f"{model_cfg.display_name} — {results_dir.name}",
)

args_payload = training_args_payload(
    model_cfg, device,
    n_neighbors_grid=KNN_NEIGHBORS_GRID, weights=KNN_WEIGHTS, metric=KNN_METRIC,
    best_n_neighbors=best_log.n_neighbors,
)
with open(results_dir / "training_args.json", "w", encoding="utf-8") as f:
    json.dump(args_payload, f, indent=2)

print(f"saved {len(results_df)} predictions + metrics report to {results_dir}")